# Notebook 45 — Optuna : Stratégie 5 EWC (Food101)

Recherche automatique des hyperparamètres via **Optuna** (20 trials, TPE sampler).

## Hyperparamètres optimisés
| Paramètre | Espace | Défaut |
|-----------|--------|--------|
| `lr_pretrain` | log-uniform [1e-4, 1e-2] | 1e-3 |
| `lr_finetune` | log-uniform [1e-5, 1e-3] | 3e-4 |
| `ewc_lambda`  | log-uniform [10, 1000] | 400 |
| `epochs_bf`   | int [5, 15] | 10 |
| `epochs_hf`   | int [5, 15] | 10 |

In [ ]:
import os, json, time, random, statistics
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _OPTUNA_AVAILABLE = True
except ImportError:
    _OPTUNA_AVAILABLE = False
    print('optuna non installe: pip install optuna')

try:
    import wandb; _WANDB_AVAILABLE = True
except ImportError:
    _WANDB_AVAILABLE = False

SEED = 42; random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Appareil: {device}')

import sys, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path: sys.path.insert(0, _p)
import degradation; importlib.reload(degradation)
from degradation import DegradedDataset, hf_transform, clean_tensor_transform
import cost; importlib.reload(cost)
from cost import data_cost
import env_config; importlib.reload(env_config)
print('Modules charges.')

In [ ]:
DATASET_NAME     = 'Food101'
BASE_DIR         = env_config.ensure_dataset_ready(DATASET_NAME)
RESULTS_DIR      = env_config.results_dir(DATASET_NAME)
os.makedirs(RESULTS_DIR, exist_ok=True)

COST_HF          = 10
COST_BF          = 1
BATCH_SIZE       = 64
NUM_WORKERS      = min(8, os.cpu_count() or 2)
HF_TRAIN_RATIO   = 0.8
N_FISHER_SAMPLES = 1000
SEEDS            = [42, 1, 2]
N_OPTUNA_TRIALS  = 20
USE_WANDB        = True
WANDB_PROJECT    = 'PF22-MultiFidelity'

DEFAULT_PARAMS = {
    'lr_pretrain': 1e-3, 'lr_finetune': 3e-4,
    'ewc_lambda': 400.0, 'epochs_bf': 10, 'epochs_hf': 10,
}

for req in ['train/BF', 'train/HF', 'test']:
    p = os.path.join(BASE_DIR, req)
    if not os.path.isdir(p): raise FileNotFoundError(f'Dossier manquant: {p}')

transform_hf    = hf_transform()
transform_clean = clean_tensor_transform()
dataset_hf_full  = datasets.ImageFolder(os.path.join(BASE_DIR, 'train/HF'), transform=transform_hf)
dataset_bf_train = DegradedDataset(datasets.ImageFolder(os.path.join(BASE_DIR, 'train/BF'), transform=transform_clean), seeded=True)
dataset_test_hf  = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_hf)
dataset_test_bf  = DegradedDataset(datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_clean), seeded=True)

NUM_CLASSES = len(dataset_hf_full.classes)
print(f'Classes ({NUM_CLASSES}): {dataset_hf_full.classes}')

In [ ]:
def create_model(num_classes=10):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes); return m.to(device)

def evaluate(model, loader, criterion):
    model.eval(); total_loss = total_correct = total_n = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x); loss = criterion(logits, y)
            total_loss += loss.item()*x.size(0)
            total_correct += (logits.argmax(1)==y).sum().item(); total_n += x.size(0)
    return total_loss/total_n, 100.0*total_correct/total_n

def make_grad_scaler():
    try:    return GradScaler('cuda', enabled=(device.type == 'cuda'))
    except TypeError: return GradScaler(enabled=(device.type == 'cuda'))

def train_phase(model, loader, val_loader, epochs, lr, phase_name,
                use_wandb=False, global_epoch_offset=0):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scaler = make_grad_scaler()
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x); loss = criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        _, val_acc = evaluate(model, val_loader, criterion)
        print(f'[{phase_name}] Ep {epoch}/{epochs}: val_acc={val_acc:.2f}%')
        if use_wandb and _WANDB_AVAILABLE:
            wandb.log({'epoch': global_epoch_offset+epoch, 'phase': phase_name,
                       'val/accuracy': val_acc})

def compute_fisher_diagonal(model, loader, n_samples=1000):
    """Diagonale Fisher (float32, sans autocast) sur n_samples images BF."""
    fisher = {n: torch.zeros_like(p.data)
              for n, p in model.named_parameters() if p.requires_grad}
    model.eval(); count = 0
    for x, y in loader:
        if count >= n_samples: break
        x, y = x.to(device), y.to(device)
        model.zero_grad()
        with torch.enable_grad():
            logits = model(x).float()
            log_probs = F.log_softmax(logits, dim=1)
            loss = F.nll_loss(log_probs, y)
            loss.backward()
        bs = x.size(0)
        for n, p in model.named_parameters():
            if p.requires_grad and p.grad is not None:
                fisher[n].data += p.grad.float().data.pow(2) * bs
        count += bs
    for n in fisher: fisher[n] /= max(count, 1)
    print(f'Fisher calculee sur {count} images.')
    return fisher

def ewc_penalty(model, fisher, params_star, ewc_lambda):
    penalty = torch.tensor(0.0, device=device)
    for n, p in model.named_parameters():
        if p.requires_grad and n in fisher and n in params_star:
            diff = p.float() - params_star[n].float()
            penalty = penalty + (fisher[n].float() * diff.pow(2)).sum()
    return ewc_lambda * penalty

def train_finetune_ewc(model, loader, val_loader, epochs, lr, fisher,
                        params_star, ewc_lambda, use_wandb=False,
                        global_epoch_offset=0):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scaler = make_grad_scaler()
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x); ce_loss = criterion(logits, y)
            ewc_val = ewc_penalty(model, fisher, params_star, ewc_lambda)
            total_loss = ce_loss.float() + ewc_val
            scaler.scale(total_loss).backward()
            scaler.step(optimizer); scaler.update()
        _, val_acc = evaluate(model, val_loader, criterion)
        print(f'[Finetune_HF+EWC] Ep {epoch}/{epochs}: val_acc={val_acc:.2f}%')
        if use_wandb and _WANDB_AVAILABLE:
            wandb.log({'epoch': global_epoch_offset+epoch,
                       'val/accuracy': val_acc})

print('Helpers EWC definis.')

In [ ]:
def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def run_with_params(seed, lr_pretrain, lr_finetune, ewc_lambda,
                     epochs_bf, epochs_hf,
                     use_wandb=False, run_name_suffix=''):
    """Entraine Strategie 5 EWC avec hyperparametres explicites."""
    set_all_seeds(seed)
    n_hf = int(HF_TRAIN_RATIO * len(dataset_hf_full))
    ds_hf_train, ds_hf_val = random_split(
        dataset_hf_full, [n_hf, len(dataset_hf_full) - n_hf],
        generator=torch.Generator().manual_seed(seed))
    ld_bf   = DataLoader(dataset_bf_train, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=True)
    ld_hf   = DataLoader(ds_hf_train, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=True)
    ld_val  = DataLoader(ds_hf_val,   batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
    ld_t_hf = DataLoader(dataset_test_hf, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
    ld_t_bf = DataLoader(dataset_test_bf, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
    track = use_wandb and _WANDB_AVAILABLE
    if track:
        wandb.init(project=WANDB_PROJECT,
                   name=f'S5_Optuna_{DATASET_NAME}_{run_name_suffix}_seed{seed}',
                   reinit=True, config={'lr_pretrain': lr_pretrain,
                       'lr_finetune': lr_finetune, 'ewc_lambda': ewc_lambda,
                       'epochs_bf': epochs_bf, 'epochs_hf': epochs_hf,
                       'dataset': DATASET_NAME, 'seed': seed})
    model = create_model(num_classes=NUM_CLASSES)
    t0 = time.time()
    # Phase A: BF pretraining
    train_phase(model, ld_bf, ld_val, epochs_bf, lr_pretrain,
              'Pretrain_BF', use_wandb=track)
    # EWC anchor
    fisher = compute_fisher_diagonal(model, ld_bf, n_samples=N_FISHER_SAMPLES)
    params_star = {n: p.data.clone()
                   for n, p in model.named_parameters() if p.requires_grad}
    # Phase B: HF finetuning with EWC penalty
    train_finetune_ewc(model, ld_hf, ld_val, epochs_hf, lr_finetune,
                        fisher, params_star, ewc_lambda,
                        use_wandb=track, global_epoch_offset=epochs_bf)
    tmin = (time.time() - t0) / 60.0
    crit = nn.CrossEntropyLoss()
    _, val_acc = evaluate(model, ld_val,   crit)
    _, hf_acc  = evaluate(model, ld_t_hf,  crit)
    _, bf_acc  = evaluate(model, ld_t_bf,  crit)
    mix_acc = (hf_acc + bf_acc) / 2.0
    total_cost = float(len(dataset_bf_train)*COST_BF*epochs_bf +
                       len(ds_hf_train)*COST_HF*epochs_hf)
    if track:
        wandb.log({'test/accuracy_HF': hf_acc, 'test/accuracy_BF': bf_acc}); wandb.finish()
    print(f'[seed {seed}] HF={hf_acc:.2f}% BF={bf_acc:.2f}% | lambda={ewc_lambda:.0f} | {tmin:.1f} min')
    return {'val_acc': val_acc, 'accuracy_HF': hf_acc, 'accuracy_BF': bf_acc,
            'accuracy_Mixte': mix_acc, 'total_cost_CA': total_cost,
            'time_min': tmin, 'n_hf_train': len(ds_hf_train), 'model': model}

def objective(trial):
    lr_pretrain = trial.suggest_float('lr_pretrain', 1e-4, 1e-2, log=True)
    lr_finetune = trial.suggest_float('lr_finetune', 1e-5, 1e-3, log=True)
    ewc_lambda  = trial.suggest_float('ewc_lambda', 10.0, 1000.0, log=True)
    epochs_bf   = trial.suggest_int('epochs_bf', 5, 15)
    epochs_hf   = trial.suggest_int('epochs_hf', 5, 15)
    r = run_with_params(42, lr_pretrain, lr_finetune, ewc_lambda, epochs_bf, epochs_hf)
    return r['val_acc']

print('Fonctions run_with_params et objective definies.')

In [ ]:
if not _OPTUNA_AVAILABLE:
    raise RuntimeError('Optuna non installe. Executer: pip install optuna')

sampler_opt = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler_opt,
                            study_name=f'strat5_{DATASET_NAME}')

print(f'Lancement de {N_OPTUNA_TRIALS} trials Optuna (Strategy 5 EWC, {DATASET_NAME})...')
t0_opt = time.time()
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
t_optuna_min = (time.time() - t0_opt) / 60.0

best_params = study.best_params
best_trial  = study.best_trial
print(f'Optuna termine en {t_optuna_min:.1f} min')
print(f'Best trial #{best_trial.number} | val_acc={best_trial.value:.2f}%')
print(f'Best params: {best_params}')

all_trials = [{
    'number': t.number, 'value': t.value if t.value is not None else float('nan'),
    'params': t.params, 'state': str(t.state)
} for t in study.trials]

optuna_results = {
    'strategy': 'elastic_weight_consolidation', 'dataset': DATASET_NAME,
    'n_trials': N_OPTUNA_TRIALS, 'optimization_seed': 42,
    'best_trial_number': best_trial.number,
    'best_val_acc_hf_pct': float(best_trial.value),
    'default_hyperparams': DEFAULT_PARAMS, 'best_params': best_params,
    'optuna_time_min': t_optuna_min, 'all_trials': all_trials,
}
params_path = os.path.join(RESULTS_DIR, 'results_strategy5_optuna_best_params.json')
with open(params_path, 'w', encoding='utf-8') as f:
    json.dump(optuna_results, f, indent=2, ensure_ascii=False)
print('Best params JSON:', params_path)

In [ ]:
print(f'Evaluation finale (3 seeds) avec best params: {best_params}')
t0_final = time.time()
per_seed_opt = [
    run_with_params(s, lr_pretrain=best_params['lr_pretrain'],
                    lr_finetune=best_params['lr_finetune'],
                    ewc_lambda=best_params['ewc_lambda'],
                    epochs_bf=best_params['epochs_bf'],
                    epochs_hf=best_params['epochs_hf'],
                    use_wandb=USE_WANDB, run_name_suffix='optimized')
    for s in SEEDS
]
model_opt = per_seed_opt[0]['model']

def _agg(key):
    vals = [r[key] for r in per_seed_opt]
    m = statistics.mean(vals); s = statistics.stdev(vals) if len(vals) > 1 else 0.0
    return m, s, vals

hf_m, hf_s, hf_all = _agg('accuracy_HF')
bf_m, bf_s, bf_all = _agg('accuracy_BF')
mx_m, mx_s, mx_all = _agg('accuracy_Mixte')
tt_m, tt_s, _      = _agg('time_min')
cost_m, _, _       = _agg('total_cost_CA')

n_hf_pool = len(dataset_hf_full); n_bf_pool = len(dataset_bf_train)
data_cost_CA = data_cost(n_hf=n_hf_pool, n_bf=n_bf_pool)

optimized_results = {
    'strategy': 'elastic_weight_consolidation', 'dataset': DATASET_NAME,
    'hyperparams_source': 'optuna_optimized', 'hyperparams': best_params,
    'default_hyperparams': DEFAULT_PARAMS, 'n_optuna_trials': N_OPTUNA_TRIALS,
    'best_optuna_val_acc': float(best_trial.value), 'seeds': SEEDS,
    'n_hf_pool': n_hf_pool, 'n_bf_pool': n_bf_pool,
    'data_cost_CA': float(data_cost_CA), 'total_cost_CA': cost_m,
    'accuracy_HF': hf_m, 'accuracy_HF_std': hf_s, 'accuracy_HF_seeds': hf_all,
    'accuracy_BF': bf_m, 'accuracy_BF_std': bf_s, 'accuracy_BF_seeds': bf_all,
    'accuracy_Mixte': mx_m, 'accuracy_Mixte_std': mx_s, 'accuracy_Mixte_seeds': mx_all,
    'total_time_min': tt_m, 'total_time_min_std': tt_s,
    'per_seed': [{k: v for k, v in r.items() if k != 'model'} for r in per_seed_opt],
}

default_path = os.path.join(RESULTS_DIR, 'results_strategy5_ewc.json')
if os.path.exists(default_path):
    with open(default_path, 'r', encoding='utf-8') as f: def_r = json.load(f)
    optimized_results['delta_accuracy_HF']    = hf_m - def_r.get('accuracy_HF', 0)
    optimized_results['delta_accuracy_BF']    = bf_m - def_r.get('accuracy_BF', 0)
    optimized_results['delta_accuracy_Mixte'] = mx_m - def_r.get('accuracy_Mixte', 0)

opt_path = os.path.join(RESULTS_DIR, 'results_strategy5_ewc_optimized.json')
pth_path = os.path.join(RESULTS_DIR, 'model_strategy5_ewc_optimized.pth')
with open(opt_path, 'w', encoding='utf-8') as f:
    json.dump(optimized_results, f, indent=2, ensure_ascii=False)
torch.save(model_opt.state_dict(), pth_path)

print(f'--- RESULTATS OPTIMISES - Strategy 5 EWC - {DATASET_NAME} ---')
print(f'Test HF    : {hf_m:.2f} +/- {hf_s:.2f} %')
print(f'Test BF    : {bf_m:.2f} +/- {bf_s:.2f} %')
print(f'Test Mixte : {mx_m:.2f} +/- {mx_s:.2f} %')
if 'delta_accuracy_HF' in optimized_results:
    print(f'Delta HF   : {optimized_results["delta_accuracy_HF"]:+.2f}%')
    print(f'Delta BF   : {optimized_results["delta_accuracy_BF"]:+.2f}%')
print('JSON:', opt_path)

In [ ]:
trial_vals = [t['value'] for t in all_trials if t['value'] == t['value']]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Strategie 5 EWC — Optuna ({DATASET_NAME}, {N_OPTUNA_TRIALS} trials)', fontsize=14)

axes[0].plot(range(1, len(trial_vals)+1), trial_vals, marker='o', markersize=4,
             linewidth=1.5, color='tab:blue')
axes[0].axhline(max(trial_vals), color='tab:red', linestyle='--',
                label=f'Best: {max(trial_vals):.2f}%')
axes[0].set_title('Progression des trials Optuna')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val acc HF (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

default_path = os.path.join(RESULTS_DIR, 'results_strategy5_ewc.json')
if os.path.exists(default_path):
    with open(default_path, 'r', encoding='utf-8') as f: def_r = json.load(f)
    metrics = ['Test HF', 'Test BF', 'Test Mixte']
    def_vals = [def_r.get('accuracy_HF',0), def_r.get('accuracy_BF',0), def_r.get('accuracy_Mixte',0)]
    def_stds = [def_r.get('accuracy_HF_std',0), def_r.get('accuracy_BF_std',0), def_r.get('accuracy_Mixte_std',0)]
    x = np.arange(len(metrics)); w = 0.3
    b1 = axes[1].bar(x - w/2, def_vals, w, label='Default', color='#E07070',
                     yerr=def_stds, capsize=4, alpha=0.85)
    b2 = axes[1].bar(x + w/2, [hf_m, bf_m, mx_m], w, label='Optimise (Optuna)',
                     color='#8172B2', yerr=[hf_s, bf_s, mx_s], capsize=4, alpha=0.85)
    for bars, vals in [(b1, def_vals), (b2, [hf_m, bf_m, mx_m])]:
        for bar, v in zip(bars, vals):
            axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
                        f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
    axes[1].set_title('Precisions: Default vs Optimise')
    axes[1].set_xticks(x); axes[1].set_xticklabels(metrics)
    axes[1].set_ylim(0, 105); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Resultats default non trouves.',
                ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout(rect=[0, 0, 1, 0.95])
fig_path = os.path.join(RESULTS_DIR, 'strategy5_optuna_comparison.png')
plt.savefig(fig_path, dpi=180, bbox_inches='tight'); plt.show()
print('Figure:', fig_path)

In [ ]:
print(f'\n=== HYPERPARAMETRES: Default vs Optuna (Strategie 5 EWC) ===')
param_names = list(best_params.keys())
hdr = f"{'Parametre':<20} {'Defaut':>15} {'Optuna':>15}"
print(hdr); print('-'*len(hdr))
for k in param_names:
    dv = DEFAULT_PARAMS.get(k, 'N/A')
    ov = best_params[k]
    dv_str = f'{dv:.2e}' if isinstance(dv, float) else str(dv)
    ov_str = f'{ov:.2e}' if isinstance(ov, float) else str(ov)
    print(f'{k:<20} {dv_str:>15} {ov_str:>15}')

comp_files = {
    'BL 1(HF)':             'results_baseline_HF.json',
    'BL 2(BF)':             'results_baseline_BF.json',
    'BL 3(MIXTE)':          'results_baseline_MIXTE.json',
    'Strat 1(default)':     'results_strategy1_transfer_learning.json',
    'Strat 1(Optuna)':      'results_strategy1_transfer_learning_optimized.json',
    'Strat 2(default)':     'results_strategy2_cotraining_reweighting.json',
    'Strat 2(Optuna)':      'results_strategy2_cotraining_reweighting_optimized.json',
    'Strat 3(default)':     'results_strategy3_curriculum_learning.json',
    'Strat 3(Optuna)':      'results_strategy3_curriculum_learning_optimized.json',
    'Strat 5(default)':     'results_strategy5_ewc.json',
    'Strat 5(Optuna)':      'results_strategy5_ewc_optimized.json',
}
loaded = {}
for name, fn in comp_files.items():
    p = os.path.join(RESULTS_DIR, fn)
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f: loaded[name] = json.load(f)

if loaded:
    print(f'\n=== RECAPITULATIF COMPLET — {DATASET_NAME} ===')
    hdr = f"{'Modele':<30} {'HF%':>7} {'BF%':>7} {'Mixte%':>9} {'Cout(CA)':>11}"
    print(hdr); print('-'*len(hdr))
    for name, d in loaded.items():
        c = int(d.get('total_cost_CA', 0))
        print(f"{name:<30} {d.get('accuracy_HF', float('nan')):>7.2f} "
              f"{d.get('accuracy_BF', float('nan')):>7.2f} "
              f"{d.get('accuracy_Mixte', float('nan')):>9.2f} {c:>11}")